In [5]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ──────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────────────────────────────────────
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

# ──────────────────────────────────────────────────────────────────────────────
# 2. TIME FEATURES (ZERO-LAG STRATEGY)
# ──────────────────────────────────────────────────────────────────────────────
def parse_time(df: pd.DataFrame) -> pd.DataFrame:
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = parts[0]
    df['minute'] = parts[1]
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    
    # Cyclical time helps the model understand the transition from night to morning
    df['sin_time'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['cos_time'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    
    # Categorical Time Slot (Crucial for CatBoost's internal heatmap)
    df['time_slot_cat'] = 'Slot_' + df['time_slot'].astype(str)
    
    # DANGER: We DO NOT return or use 'day' because it will crash the test set!
    return df

train = parse_time(train)
test  = parse_time(test)

# ──────────────────────────────────────────────────────────────────────────────
# 3. IMPUTATION & CATEGORICALS
# ──────────────────────────────────────────────────────────────────────────────
# Fill temperatures with outlier flags
train['Temperature'] = train['Temperature'].fillna(-999)
test['Temperature']  = test['Temperature'].fillna(-999)

CAT_FEATURES = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'time_slot_cat']
for col in CAT_FEATURES:
    train[col] = train[col].fillna('Unknown').astype(str)
    test[col]  = test[col].fillna('Unknown').astype(str)

# ──────────────────────────────────────────────────────────────────────────────
# 4. FEATURES (Notice: 'day' and 'lags' are explicitly excluded!)
# ──────────────────────────────────────────────────────────────────────────────
FEATURES = [
    'geohash', 'hour', 'minute', 'time_slot', 'time_slot_cat',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather', 'sin_time', 'cos_time'
]
TARGET = 'demand'

# Since we don't rely on chronological lags anymore, a random split is the mathematically correct way to validate.
X_train, X_val, y_train, y_val = train_test_split(
    train[FEATURES], train[TARGET], test_size=0.15, random_state=42
)

# ──────────────────────────────────────────────────────────────────────────────
# 5. CATBOOST TRAINING (Deep & Highly Regularized)
# ──────────────────────────────────────────────────────────────────────────────
MODEL_PARAMS = dict(
    iterations   = 5000,    # Let it run longer
    learning_rate= 0.03,    # Slower learning for stability
    depth        = 8,       # Deeper trees to catch geohash x time interactions
    l2_leaf_reg  = 5,       # Higher regularization to prevent memorizing
    loss_function= 'RMSE',
    eval_metric  = 'RMSE',
    od_type      = 'Iter',
    od_wait      = 250,
    verbose      = 500,
)

print("Training Zero-Lag Baseline Model...")
model = CatBoostRegressor(**MODEL_PARAMS, random_seed=42)
model.fit(X_train, y_train, cat_features=CAT_FEATURES, eval_set=(X_val, y_val))

val_preds = np.clip(model.predict(X_val), 0, None)
r2        = r2_score(y_val, val_preds)
print(f"\nValidation R²: {r2:.5f} | Estimated Score: {max(0.0, 100.0 * r2):.2f}/100")

# ──────────────────────────────────────────────────────────────────────────────
# 6. ENSEMBLE RETRAIN ON 100% DATA
# ──────────────────────────────────────────────────────────────────────────────
print("\nTraining 3-Seed Final Ensemble...")
best_iters = model.best_iteration_
SEEDS = [42, 123, 2024]
test_preds = np.zeros(len(test))

for i, seed in enumerate(SEEDS):
    print(f"  Training Seed {i+1}/3...")
    seed_model = CatBoostRegressor(**{**MODEL_PARAMS, 'iterations': best_iters, 'random_seed': seed, 'od_type': 'IncToDec', 'od_wait': 10, 'verbose': 0})
    seed_model.fit(train[FEATURES], train[TARGET], cat_features=CAT_FEATURES)
    test_preds += seed_model.predict(test[FEATURES]) / len(SEEDS)

test_preds = np.clip(test_preds, 0, None)
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
submission.to_csv('submission_zero_lag.csv', index=False)
print("\nSaved to submission_zero_lag.csv")

Training Zero-Lag Baseline Model...
0:	learn: 0.1390070	test: 0.1390901	best: 0.1390901 (0)	total: 91.1ms	remaining: 7m 35s
500:	learn: 0.0387418	test: 0.0389716	best: 0.0389714 (499)	total: 54s	remaining: 8m 4s
1000:	learn: 0.0352023	test: 0.0366703	best: 0.0366691 (999)	total: 2m 12s	remaining: 8m 50s
1500:	learn: 0.0329769	test: 0.0354669	best: 0.0354669 (1500)	total: 3m 1s	remaining: 7m 2s
2000:	learn: 0.0313110	test: 0.0346333	best: 0.0346327 (1998)	total: 4m 6s	remaining: 6m 9s
2500:	learn: 0.0301722	test: 0.0342040	best: 0.0342036 (2499)	total: 5m 16s	remaining: 5m 16s
3000:	learn: 0.0293238	test: 0.0339272	best: 0.0339272 (3000)	total: 6m 5s	remaining: 4m 3s
3500:	learn: 0.0284806	test: 0.0336743	best: 0.0336743 (3500)	total: 6m 54s	remaining: 2m 57s
4000:	learn: 0.0277742	test: 0.0334767	best: 0.0334764 (3996)	total: 7m 44s	remaining: 1m 55s
4500:	learn: 0.0271509	test: 0.0333387	best: 0.0333387 (4498)	total: 8m 34s	remaining: 57s
4999:	learn: 0.0266036	test: 0.0332370	best: 0